# SmartDemand: leak-free evaluation

The final notebook (`02_Final_Model_CategoryLevel.ipynb`) reports R² 0.9541 and MAE 8.92 on a **random** split. That number has two problems:

1. **Leakage.** `num_products` counts the products sold in the same month the model is predicting, so it is only known after the month ends. It carried 49% of the feature importance. `seasonality_index` was also computed over the whole dataset, test months included.
2. **Random split on time-ordered data.** The model trains on months that come after the months it is tested on.

This notebook repeats the data preparation from notebook 02, then evaluates on a **time-ordered split** (the last ~20% of months as test) using **only features known before the month starts**: last month's sales, the category, a seasonality index computed on training months, and last month's price, freight, rating, product count, price range and discount rate.

The baseline is the naive forecast *next month = last month*.


## 1. Data preparation (same as notebook 02)

In [ ]:
!pip install -q pandas numpy scikit-learn matplotlib seaborn joblib

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import json
import os
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

plt.style.use('seaborn-v0_8-whitegrid')

import sklearn
print(f'pandas       : {pd.__version__}')
print(f'scikit-learn : {sklearn.__version__}')
print('Libraries loaded.')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DATA_PATH  = '/content/drive/MyDrive/dataset/'
MODEL_PATH = '/content/drive/MyDrive/SmartDemand_Dataset/models/'
os.makedirs(MODEL_PATH, exist_ok=True)
print('Drive mounted.')

In [ ]:
df_orders     = pd.read_csv(DATA_PATH + 'olist_orders_dataset.csv')
df_items      = pd.read_csv(DATA_PATH + 'olist_order_items_dataset.csv')
df_products   = pd.read_csv(DATA_PATH + 'olist_products_dataset.csv')
df_reviews    = pd.read_csv(DATA_PATH + 'olist_order_reviews_dataset.csv')
df_payments   = pd.read_csv(DATA_PATH + 'olist_order_payments_dataset.csv')
df_cat_transl = pd.read_csv(DATA_PATH + 'product_category_name_translation.csv')

print('Loaded:')
for name, df in [('orders',df_orders),('items',df_items),
                  ('products',df_products),('reviews',df_reviews),
                  ('payments',df_payments),('cat_transl',df_cat_transl)]:
    print(f'  {name:12s}: {df.shape}')

In [ ]:
# Aggregate reviews per order
review_avg = (
    df_reviews
    .groupby('order_id')['review_score']
    .mean().reset_index()
    .rename(columns={'review_score': 'product_rating'})
)

# Aggregate payments per order
payment_per_order = (
    df_payments
    .groupby('order_id')['payment_value']
    .sum().reset_index()
    .rename(columns={'payment_value': 'total_payment'})
)

# Build master dataset
master = df_items.copy()
master = master.merge(
    df_orders[['order_id','order_purchase_timestamp','order_status']],
    on='order_id', how='left'
)
master = master.merge(review_avg, on='order_id', how='left')
master = master.merge(payment_per_order, on='order_id', how='left')
master = master.merge(
    df_products[['product_id','product_category_name']],
    on='product_id', how='left'
)
master = master.merge(df_cat_transl, on='product_category_name', how='left')

# Use English category name
master['category'] = (
    master['product_category_name_english']
    .fillna(master['product_category_name'])
    .fillna('unknown')
)

# Filter delivered orders only
master = master[master['order_status'] == 'delivered'].copy()

# Parse dates
master['order_purchase_timestamp'] = pd.to_datetime(master['order_purchase_timestamp'])
master['year']  = master['order_purchase_timestamp'].dt.year
master['month'] = master['order_purchase_timestamp'].dt.month

# Item total per order for discount calculation
order_item_total = (
    master.groupby('order_id')['price']
    .sum().reset_index()
    .rename(columns={'price': 'item_total'})
)
master = master.merge(order_item_total, on='order_id', how='left')

print(f'Master dataset shape: {master.shape}')
print(f'Date range: {master["year"].min()} – {master["year"].max()}')

In [ ]:
# Aggregate to category-month level
df_agg = (
    master
    .groupby(['category', 'year', 'month'])
    .agg(
        quantity_sold   = ('order_item_id', 'count'),
        avg_price       = ('price', 'mean'),
        avg_freight     = ('freight_value', 'mean'),
        avg_rating      = ('product_rating', 'mean'),
        avg_item_total  = ('item_total', 'mean'),
        avg_payment     = ('total_payment', 'mean'),
        num_products    = ('product_id', 'nunique'),       # product variety in category
        min_price       = ('price', 'min'),
        max_price       = ('price', 'max'),
    )
    .reset_index()
)

print(f'Category-level dataset shape: {df_agg.shape}')
print(f'Unique categories: {df_agg["category"].nunique()}')
print(f'\nSample quantity_sold range:')
print(f'  Min    : {df_agg["quantity_sold"].min()}')
print(f'  Median : {df_agg["quantity_sold"].median():.0f}')
print(f'  Max    : {df_agg["quantity_sold"].max()}')
display(df_agg.head(5))

In [ ]:
df_clean = df_agg.copy()

# Fill missing ratings with overall median
df_clean['avg_rating'] = df_clean['avg_rating'].fillna(df_clean['avg_rating'].median())

# Fill missing payment with item total
df_clean['avg_payment'] = df_clean['avg_payment'].fillna(df_clean['avg_item_total'])

# Drop rows with missing price or freight
df_clean = df_clean.dropna(subset=['avg_price', 'avg_freight'])

# Remove extreme outliers (3x IQR)
q1  = df_clean['quantity_sold'].quantile(0.25)
q3  = df_clean['quantity_sold'].quantile(0.75)
iqr = q3 - q1
df_clean = df_clean[df_clean['quantity_sold'] <= q3 + 3 * iqr]

print(f'Rows after cleaning : {len(df_clean):,}')
print(f'Missing values      : {df_clean.isnull().sum().sum()}')
print(f'\nQuantity sold stats:')
print(df_clean['quantity_sold'].describe().round(1))

In [ ]:
df_feat = df_clean.copy()

df_feat['discount_rate'] = (
    (df_feat['avg_item_total'] - df_feat['avg_payment'])
    / df_feat['avg_item_total'].replace(0, np.nan)
).clip(lower=0).fillna(0)

df_feat['price_range'] = df_feat['max_price'] - df_feat['min_price']

monthly_avg  = df_feat.groupby('month')['quantity_sold'].mean()
monthly_norm = (monthly_avg - monthly_avg.min()) / (monthly_avg.max() - monthly_avg.min())
df_feat['seasonality_index'] = df_feat['month'].map(monthly_norm)

df_feat = df_feat.sort_values(['category', 'year', 'month'])
df_feat['last_month_sales'] = (
    df_feat.groupby('category')['quantity_sold']
    .shift(1).fillna(0)
)

label_enc = LabelEncoder()
df_feat['category_encoded'] = label_enc.fit_transform(df_feat['category'])

print('Feature engineering complete.')

In [ ]:
FEATURES = [
    'avg_price',
    'discount_rate',
    'avg_freight',
    'avg_rating',
    'num_products',
    'price_range',
    'category_encoded',
    'seasonality_index',
    'last_month_sales',
]
TARGET = 'quantity_sold'

print(f'Features ({len(FEATURES)}) : {FEATURES}')
print(f'Target              : {TARGET}')
print(f'Dataset shape       : {df_feat[FEATURES + [TARGET]].shape}')
display(df_feat[FEATURES + [TARGET]].describe().round(2))

## 2. Leak-free model on a time-ordered split

Rows are kept only when the previous row of the same category really is the previous month, so *last month* means last month.

In [ ]:
# Leak-free check. Run after the feature engineering cell (df_feat must exist).
# Time-ordered split, and only features you would have before the month starts.
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score

d = df_feat.copy()
d['period'] = d['year'] * 12 + d['month']
d = d.sort_values(['category', 'period'])
g = d.groupby('category')

# Keep rows whose previous row really is last month, so "last month" means last month.
d['gap'] = d['period'] - g['period'].shift(1)
SAME_MONTH = ['avg_price', 'avg_freight', 'avg_rating', 'num_products', 'price_range', 'discount_rate']
for col in SAME_MONTH:
    d['prev_' + col] = g[col].shift(1)
d = d[d['gap'] == 1].copy()

# Train on earlier months, test on the last ~20% of months.
cutoff = d['period'].quantile(0.8)
train = d[d['period'] < cutoff].copy()
test = d[d['period'] >= cutoff].copy()

# Seasonality from training months only.
m = train.groupby('month')['quantity_sold'].mean()
m = (m - m.min()) / (m.max() - m.min())
for part in (train, test):
    part['season_train'] = part['month'].map(m).fillna(m.mean())

OLD = FEATURES  # original 9 features, several only known after the month ends
NEW = ['last_month_sales', 'category_encoded', 'season_train'] + ['prev_' + c for c in SAME_MONTH]

def run(cols):
    rf = RandomForestRegressor(n_estimators=300, max_depth=20, min_samples_split=5,
                               min_samples_leaf=2, max_features='sqrt', random_state=42, n_jobs=-1)
    rf.fit(train[cols], train['quantity_sold'])
    return rf, rf.predict(test[cols])

y = test['quantity_sold']
naive = test['last_month_sales']
rf_old, p_old = run(OLD)
rf_new, p_new = run(NEW)

print(f'Train rows {len(train)}, test rows {len(test)}, test months {test["period"].nunique()}')
print(f'{"model":34s} {"MAE":>7s} {"R2":>7s}')
for name, p in [('Naive: next month = last month', naive),
                ('RF, original features', p_old),
                ('RF, leak-free features', p_new)]:
    print(f'{name:34s} {mean_absolute_error(y, p):7.2f} {r2_score(y, p):7.4f}')
print(f'Median units per category-month in test: {y.median():.0f}')
print()
print(pd.Series(rf_new.feature_importances_, index=NEW).sort_values(ascending=False).round(4).to_string())

### Result (Colab run, September 2026)

| Model | MAE | R² |
|---|---|---|
| Naive: next month = last month | **11.63** | 0.9361 |
| Random Forest, original features (leaky) | 9.47 | 0.9541 |
| Random Forest, leak-free features | 16.41 | 0.8777 |

225 test rows over the last 4 months; median 23 units per category-month. Once the leaky features are removed, the naive baseline beats the model. In the leak-free model, last month's sales carry 49% of the importance and last month's product count 32%.

## 3. Second try: predict the change from last month

Same features, but the target is the change in units from last month; the forecast is last month plus the predicted change.

In [ ]:
# Run right after the leak-free cell (it reuses train, test and NEW).
# Same features, but the model predicts the CHANGE from last month,
# and the forecast is last month + that change.
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score

delta_train = train['quantity_sold'] - train['last_month_sales']

rf_delta = RandomForestRegressor(n_estimators=300, max_depth=20, min_samples_split=5,
                                 min_samples_leaf=2, max_features='sqrt', random_state=42, n_jobs=-1)
rf_delta.fit(train[NEW], delta_train)
p_delta = test['last_month_sales'] + rf_delta.predict(test[NEW])

y = test['quantity_sold']
naive = test['last_month_sales']
for name, p in [('Naive: next month = last month', naive),
                ('RF, leak-free, predicts total', p_new),
                ('RF, leak-free, predicts change', p_delta)]:
    print(f'{name:34s} MAE {mean_absolute_error(y, p):7.2f}  R2 {r2_score(y, p):7.4f}')

# How often each one is closer, row by row
closer = (abs(y - p_delta) < abs(y - naive)).mean()
print(f'Change model closer than naive on {closer:.0%} of test rows')

### Result (Colab run, September 2026)

| Model | MAE | R² |
|---|---|---|
| Naive: next month = last month | **11.63** | 0.9361 |
| Random Forest, leak-free, predicts total | 16.41 | 0.8777 |
| Random Forest, leak-free, predicts change | 17.98 | 0.8555 |

The change model is closer than naive on only 30% of test rows. Conclusion: with roughly two years of monthly data per category, *same as last month* is the forecast to beat, and neither model beats it yet. Next steps: more months of history, and keep a model only if it beats 11.63 on a time-ordered split.